# 09 — Model Training & Evaluation
**Fingo Income Predictor** | Tim CC26-PSU217 | DS2 Clarisya Adeline

## Input
- `outputs/model_contract/income_train.csv`
- `outputs/model_contract/income_val.csv`
- `outputs/model_contract/income_test.csv`
- `outputs/model_contract/feature_columns.json`
- `outputs/model_contract/income_scalers.pkl`
- `outputs/model_contract/model_contract.json`

## Output
- `outputs/model_results/best_income_regressor.pkl`
- `outputs/model_results/best_direction_classifier.pkl`
- `outputs/model_results/regression_metrics.csv`
- `outputs/model_results/classification_metrics.csv`
- `outputs/model_results/predictions_test.csv`
- `outputs/model_results/model_evaluation_report.md`
- `outputs/charts/regression_prediction_vs_actual.png`
- `outputs/charts/regression_residual_distribution.png`
- `outputs/charts/classification_confusion_matrix.png`
- `outputs/charts/feature_importance_best_model.png`
- `outputs/charts/regression_error_by_gig_type.png`


## CELL 09.1 — Git Pull Header

In [1]:
# GIT PULL — Sinkronisasi terbaru dari remote sebelum mulai
import os, shutil, subprocess

try:
    from google.colab import userdata
except Exception:
    userdata = None

os.chdir('/content')

GITHUB_USERNAME = 'ClarisyaA'
REPO_NAME       = 'fingo-income-analysis'
BRANCH_NAME     = 'feat/income-predictor-final'
LOCAL_DIR       = f'/content/{REPO_NAME}'
FRESH_CLONE     = False

def get_remote_url():
    try:
        token = userdata.get('GITHUB_TOKEN') if userdata else os.environ.get('GITHUB_TOKEN', '')
        if token:
            return f'https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git', token
    except Exception:
        pass
    return f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git', None

remote_url, token = get_remote_url()

def mask_cmd(cmd):
    return cmd.replace(token, '***TOKEN***') if token else cmd

def run_cmd(cmd, check=True, cwd='/content'):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    print(f'$ {mask_cmd(cmd)}')
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f'Command gagal: {mask_cmd(cmd)}')
    return r

def remote_branch_exists():
    r = run_cmd(f'git ls-remote --heads {remote_url} {BRANCH_NAME}', check=False)
    return r.stdout.strip() != ''

branch_exists = remote_branch_exists()

if FRESH_CLONE and os.path.exists(LOCAL_DIR):
    os.chdir('/content')
    shutil.rmtree(LOCAL_DIR)

if not os.path.exists(LOCAL_DIR):
    if branch_exists:
        run_cmd(f'git clone -b {BRANCH_NAME} {remote_url} {LOCAL_DIR}')
    else:
        run_cmd(f'git clone {remote_url} {LOCAL_DIR}')
        run_cmd(f'git checkout -b {BRANCH_NAME}', cwd=LOCAL_DIR)
else:
    run_cmd(f'git remote set-url origin {remote_url}', cwd=LOCAL_DIR)
    run_cmd('git fetch origin', cwd=LOCAL_DIR)

    if branch_exists:
        local_b = run_cmd(f'git branch --list {BRANCH_NAME}', check=False, cwd=LOCAL_DIR).stdout.strip()
        if local_b:
            run_cmd(f'git checkout {BRANCH_NAME}', cwd=LOCAL_DIR)
        else:
            run_cmd(f'git checkout -b {BRANCH_NAME} origin/{BRANCH_NAME}', cwd=LOCAL_DIR)
        run_cmd(f'git pull --rebase origin {BRANCH_NAME}', cwd=LOCAL_DIR)
    else:
        current_branch = run_cmd('git branch --show-current', check=False, cwd=LOCAL_DIR).stdout.strip()
        if current_branch != BRANCH_NAME:
            local_b = run_cmd(f'git branch --list {BRANCH_NAME}', check=False, cwd=LOCAL_DIR).stdout.strip()
            if local_b:
                run_cmd(f'git checkout {BRANCH_NAME}', cwd=LOCAL_DIR)
            else:
                run_cmd(f'git checkout -b {BRANCH_NAME}', cwd=LOCAL_DIR)

os.chdir(LOCAL_DIR)
run_cmd(f'git remote set-url origin {remote_url}', cwd=LOCAL_DIR)
print('\nRepo siap digunakan')
print(f'Working directory: {os.getcwd()}')
run_cmd('git branch --show-current', cwd=LOCAL_DIR)
run_cmd('git status --short', check=False, cwd=LOCAL_DIR)


$ git ls-remote --heads https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git feat/income-predictor-final
b5fc6991c7d90d7e6131a3de331cc81d30f9cf21	refs/heads/feat/income-predictor-final
$ git clone -b feat/income-predictor-final https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git /content/fingo-income-analysis
Cloning into '/content/fingo-income-analysis'...
Updating files:   5% (4/71)
Updating files:   7% (5/71)
Updating files:   8% (6/71)
Updating files:   9% (7/71)
Updating files:  11% (8/71)
Updating files:  12% (9/71)
Updating files:  14% (10/71)
Updating files:  15% (11/71)
Updating files:  16% (12/71)
Updating files:  18% (13/71)
Updating files:  19% (14/71)
Updating files:  21% (15/71)
Updating files:  22% (16/71)
Updating files:  23% (17/71)
Updating files:  25% (18/71)
Updating files:  26% (19/71)
Updating files:  28% (20/71)
Updating files:  29% (21/71)
Updating files:  30% (22/71)
Updating files:  32% (23/71)
Updating files:  33% (24/71)
Updating

CompletedProcess(args='git status --short', returncode=0, stdout='', stderr='')

## CELL 09.2 — Setup & Import Library

In [2]:
# CELL 09.2 — Setup & Import Library
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'scikit-learn', 'xgboost', 'pandas', 'numpy',
    'matplotlib', 'seaborn', '--quiet'
])

import os, json, pickle, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)
os.chdir('/content/fingo-income-analysis')

def ensure_dir(path):
    d = os.path.dirname(path) if '.' in os.path.basename(path) else path
    if d:
        os.makedirs(d, exist_ok=True)

def safe_to_csv(df, path, **kwargs):
    ensure_dir(path)
    df.to_csv(path, index=kwargs.pop('index', False), **kwargs)

def safe_savefig(path, dpi=150):
    ensure_dir(path)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print(f'Chart disimpan: {path}')

def fmt_idr(value):
    try:
        return f'Rp {int(round(value)):,}'.replace(',', '.')
    except Exception:
        return str(value)

def evaluate_regression(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    r2   = r2_score(y_true, y_pred)
    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2}

def evaluate_classification(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return {'accuracy': acc, 'macro_precision': prec, 'macro_recall': rec, 'macro_f1': f1}

print('Setup selesai')
print(f'Working directory: {os.getcwd()}')


Setup selesai
Working directory: /content/fingo-income-analysis


## CELL 09.3 — Load Dataset dan Model Contract

In [3]:
# CELL 09.3 — Load Dataset dan Model Contract
df_train = pd.read_csv('outputs/model_contract/income_train.csv')
df_val   = pd.read_csv('outputs/model_contract/income_val.csv')
df_test  = pd.read_csv('outputs/model_contract/income_test.csv')

with open('outputs/model_contract/feature_columns.json', encoding='utf-8') as f:
    feature_meta = json.load(f)
FEATURE_COLS = feature_meta['feature_columns']
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_train.columns]

with open('outputs/model_contract/model_contract.json', encoding='utf-8') as f:
    model_contract = json.load(f)

with open('outputs/model_contract/income_scalers.pkl', 'rb') as f:
    scalers = pickle.load(f)
target_scaler  = scalers['target_scaler']
feature_scaler = scalers['feature_scaler']

# Validasi
assert len(df_train) > 0, 'df_train kosong'
assert len(df_val)   > 0, 'df_val kosong'
assert len(df_test)  > 0, 'df_test kosong'
assert 'next_week_income' in df_train.columns, 'next_week_income tidak ditemukan'
assert 'next_week_direction' in df_train.columns, 'next_week_direction tidak ditemukan'
missing_feat = [c for c in FEATURE_COLS if c not in df_train.columns]
assert len(missing_feat) == 0, f'Feature columns tidak tersedia: {missing_feat}'

FORBIDDEN = [
    'next_week_income', 'next_week_direction', 'monthly_income',
    'avg_weekly_income', 'income_std_4w', 'income_cv_4w',
    'income_range_4w', 'income_w1', 'income_w2', 'income_w3', 'income_w4',
    'synthetic_weekly_income',
]
leaked = [c for c in FORBIDDEN if c in FEATURE_COLS]
assert len(leaked) == 0, f'LEAKAGE DETECTED: {leaked}'

print(f'Train  : {len(df_train):,} baris')
print(f'Val    : {len(df_val):,} baris')
print(f'Test   : {len(df_test):,} baris')
print(f'Feature columns: {len(FEATURE_COLS)}')
print('Anti-leakage check: PASSED')
print('Dataset dan scaler berhasil dimuat')


Train  : 100,800 baris
Val    : 21,600 baris
Test   : 21,600 baris
Feature columns: 58
Anti-leakage check: PASSED
Dataset dan scaler berhasil dimuat


## CELL 09.4 — Prepare X/y

In [4]:
# CELL 09.4 — Prepare X/y

y_train_reg = df_train['next_week_income'].values
y_val_reg   = df_val['next_week_income'].values
y_test_reg  = df_test['next_week_income'].values

y_train_reg_log = np.log1p(y_train_reg)
y_val_reg_log   = np.log1p(y_val_reg)
y_test_reg_log  = np.log1p(y_test_reg)

X_train = df_train[FEATURE_COLS].fillna(0)
X_val   = df_val[FEATURE_COLS].fillna(0)
X_test  = df_test[FEATURE_COLS].fillna(0)

X_train_scaled = feature_scaler.transform(X_train)
X_val_scaled   = feature_scaler.transform(X_val)
X_test_scaled  = feature_scaler.transform(X_test)

DIRECTION_LABEL_MAP = {'Down': 0, 'Stable': 1, 'Up': 2}
INVERSE_DIRECTION_LABEL_MAP = {0: 'Down', 1: 'Stable', 2: 'Up'}

y_train_cls = df_train['next_week_direction'].map(DIRECTION_LABEL_MAP).values
y_val_cls   = df_val['next_week_direction'].map(DIRECTION_LABEL_MAP).values
y_test_cls  = df_test['next_week_direction'].map(DIRECTION_LABEL_MAP).values

print(f'X_train shape : {X_train.shape}')
print(f'X_val shape   : {X_val.shape}')
print(f'X_test shape  : {X_test.shape}')
print('Distribusi kelas train:')
unique, counts = np.unique(y_train_cls, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {INVERSE_DIRECTION_LABEL_MAP[u]}: {c:,} ({c/len(y_train_cls)*100:.1f}%)')


X_train shape : (100800, 58)
X_val shape   : (21600, 58)
X_test shape  : (21600, 58)
Distribusi kelas train:
  Down: 12,100 (12.0%)
  Stable: 64,567 (64.1%)
  Up: 24,133 (23.9%)


## CELL 09.5 — Train Regression Baseline Models

In [5]:
# CELL 09.5 — Train Regression Baseline Models

LINEAR_MODELS_REG = {'LinearRegression', 'Ridge'}

regression_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'RandomForestRegressor': RandomForestRegressor(
        n_estimators=200, max_depth=12, random_state=42, n_jobs=-1
    ),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42),
    'XGBRegressor': XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        objective='reg:squarederror', random_state=42, n_jobs=-1,
        verbosity=0
    ),
}

trained_regressors = {}
reg_val_metrics    = []

for name, model in regression_models.items():
    try:
        print(f'Training {name} ...')
        if name in LINEAR_MODELS_REG:
            model.fit(X_train_scaled, y_train_reg_log)
            pred_log = model.predict(X_val_scaled)
        else:
            model.fit(X_train.values, y_train_reg_log)
            pred_log = model.predict(X_val.values)
        pred_idr = np.clip(np.expm1(pred_log), 0, None)
        metrics  = evaluate_regression(y_val_reg, pred_idr)
        metrics['model_name'] = name
        reg_val_metrics.append(metrics)
        trained_regressors[name] = model
        print(f'  VAL  MAE={fmt_idr(metrics["mae"])}  RMSE={fmt_idr(metrics["rmse"])}  '
              f'MAPE={metrics["mape"]:.2f}%  R2={metrics["r2"]:.4f}')
    except Exception as e:
        print(f'  [WARNING] {name} gagal dilatih: {e}')

assert len(trained_regressors) > 0, 'Tidak ada regressor yang berhasil dilatih'

df_reg_val = pd.DataFrame(reg_val_metrics).sort_values('mae').reset_index(drop=True)
print('\n=== Validation Regression Metrics (diurutkan MAE) ===')
print(df_reg_val[['model_name','mae','rmse','mape','r2']].to_string(index=False))

best_reg_name  = df_reg_val.iloc[0]['model_name']
best_regressor = trained_regressors[best_reg_name]
print(f'\nBest Regressor: {best_reg_name}')


Training LinearRegression ...
  VAL  MAE=Rp 108.027  RMSE=Rp 201.323  MAPE=125.93%  R2=0.5166
Training Ridge ...
  VAL  MAE=Rp 108.027  RMSE=Rp 201.323  MAPE=125.93%  R2=0.5166
Training RandomForestRegressor ...
  VAL  MAE=Rp 65.075  RMSE=Rp 124.156  MAPE=109.47%  R2=0.8161
Training GradientBoostingRegressor ...
  VAL  MAE=Rp 64.266  RMSE=Rp 121.263  MAPE=126.58%  R2=0.8246
Training XGBRegressor ...
  VAL  MAE=Rp 63.173  RMSE=Rp 121.460  MAPE=119.95%  R2=0.8240

=== Validation Regression Metrics (diurutkan MAE) ===
               model_name           mae          rmse       mape       r2
             XGBRegressor  63173.287168 121459.881373 119.945755 0.824040
GradientBoostingRegressor  64265.994822 121263.342519 126.583862 0.824609
    RandomForestRegressor  65074.589110 124156.318622 109.467556 0.816141
                    Ridge 108027.233342 201322.602254 125.930101 0.516571
         LinearRegression 108027.305350 201323.396066 125.927498 0.516567

Best Regressor: XGBRegressor


## CELL 09.6 — Evaluate Best Regressor on Test Set

In [6]:
# CELL 09.6 — Evaluate Best Regressor on Test Set

if best_reg_name in LINEAR_MODELS_REG:
    pred_test_log = best_regressor.predict(X_test_scaled)
else:
    pred_test_log = best_regressor.predict(X_test.values)

pred_test_idr = np.clip(np.expm1(pred_test_log), 0, None)

reg_test_metrics = evaluate_regression(y_test_reg, pred_test_idr)
print('=== Test Regression Metrics ===')
print(f'  MAE  : {fmt_idr(reg_test_metrics["mae"])}')
print(f'  RMSE : {fmt_idr(reg_test_metrics["rmse"])}')
print(f'  MAPE : {reg_test_metrics["mape"]:.2f}%')
print(f'  R2   : {reg_test_metrics["r2"]:.4f}')

id_cols  = ['synthetic_user_id']
opt_cols = ['target_week_index', 'target_date', 'gig_type']
df_test_pred = df_test[id_cols + [c for c in opt_cols if c in df_test.columns]].copy()
df_test_pred['next_week_income']           = y_test_reg
df_test_pred['predicted_next_week_income'] = pred_test_idr
df_test_pred['absolute_error']             = np.abs(y_test_reg - pred_test_idr)
mask = y_test_reg != 0
ape  = np.where(mask, np.abs(y_test_reg - pred_test_idr) / y_test_reg * 100, np.nan)
df_test_pred['absolute_percentage_error']  = ape
df_test_pred['next_week_direction']        = df_test['next_week_direction'].values

print(f'df_test_pred shape: {df_test_pred.shape}')
print(df_test_pred.head(3).to_string())


=== Test Regression Metrics ===
  MAE  : Rp 64.557
  RMSE : Rp 125.159
  MAPE : 113.81%
  R2   : 0.8279
df_test_pred shape: (21600, 9)
  synthetic_user_id  target_week_index target_date         gig_type  next_week_income  predicted_next_week_income  absolute_error  absolute_percentage_error next_week_direction
0        SYN_000001                  5  2026-01-29  content_creator           78536.0                65509.882812    13026.117188                  16.586173                  Up
1        SYN_000001                  6  2026-02-05  content_creator           82384.0                69096.929688    13287.070312                  16.128217              Stable
2        SYN_000001                  7  2026-02-12  content_creator           81842.0                70163.179688    11678.820312                  14.269960              Stable


## CELL 09.7 — Train Classification Baseline Models

In [7]:
# CELL 09.7 — Train Classification Baseline Models

LINEAR_MODELS_CLS = {'LogisticRegression'}

classification_models = {
    'LogisticRegression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'RandomForestClassifier': RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=42,
        n_jobs=-1, class_weight='balanced'
    ),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'XGBClassifier': XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        objective='multi:softmax', num_class=3,
        random_state=42, n_jobs=-1, verbosity=0,
        use_label_encoder=False, eval_metric='mlogloss'
    ),
}

trained_classifiers = {}
cls_val_metrics     = []

for name, model in classification_models.items():
    try:
        print(f'Training {name} ...')
        if name in LINEAR_MODELS_CLS:
            model.fit(X_train_scaled, y_train_cls)
            pred_val_cls = model.predict(X_val_scaled)
        else:
            model.fit(X_train.values, y_train_cls)
            pred_val_cls = model.predict(X_val.values)
        metrics = evaluate_classification(y_val_cls, pred_val_cls)
        metrics['model_name'] = name
        cls_val_metrics.append(metrics)
        trained_classifiers[name] = model
        print(f'  VAL  Acc={metrics["accuracy"]:.4f}  '
              f'Prec={metrics["macro_precision"]:.4f}  '
              f'Rec={metrics["macro_recall"]:.4f}  '
              f'F1={metrics["macro_f1"]:.4f}')
    except Exception as e:
        print(f'  [WARNING] {name} gagal dilatih: {e}')

assert len(trained_classifiers) > 0, 'Tidak ada classifier yang berhasil dilatih'

df_cls_val = pd.DataFrame(cls_val_metrics).sort_values('macro_f1', ascending=False).reset_index(drop=True)
print('\n=== Validation Classification Metrics (diurutkan macro_f1) ===')
print(df_cls_val[['model_name','accuracy','macro_precision','macro_recall','macro_f1']].to_string(index=False))

best_cls_name   = df_cls_val.iloc[0]['model_name']
best_classifier = trained_classifiers[best_cls_name]
print(f'\nBest Classifier: {best_cls_name}')


Training LogisticRegression ...
  VAL  Acc=0.7117  Prec=0.6183  Rec=0.6335  F1=0.6199
Training RandomForestClassifier ...
  VAL  Acc=0.7720  Prec=0.6694  Rec=0.6393  F1=0.6506
Training GradientBoostingClassifier ...
  VAL  Acc=0.7990  Prec=0.7617  Rec=0.6057  F1=0.6353
Training XGBClassifier ...
  VAL  Acc=0.8029  Prec=0.7705  Rec=0.6108  F1=0.6418

=== Validation Classification Metrics (diurutkan macro_f1) ===
                model_name  accuracy  macro_precision  macro_recall  macro_f1
    RandomForestClassifier  0.771991         0.669433      0.639313  0.650645
             XGBClassifier  0.802917         0.770546      0.610822  0.641785
GradientBoostingClassifier  0.798981         0.761692      0.605702  0.635287
        LogisticRegression  0.711713         0.618347      0.633468  0.619870

Best Classifier: RandomForestClassifier


## CELL 09.8 — Evaluate Best Classifier on Test Set

In [8]:
# CELL 09.8 — Evaluate Best Classifier on Test Set

if best_cls_name in LINEAR_MODELS_CLS:
    pred_test_cls = best_classifier.predict(X_test_scaled)
else:
    pred_test_cls = best_classifier.predict(X_test.values)

cls_test_metrics = evaluate_classification(y_test_cls, pred_test_cls)
print('=== Test Classification Metrics ===')
print(f'  Accuracy       : {cls_test_metrics["accuracy"]:.4f}')
print(f'  Macro Precision: {cls_test_metrics["macro_precision"]:.4f}')
print(f'  Macro Recall   : {cls_test_metrics["macro_recall"]:.4f}')
print(f'  Macro F1       : {cls_test_metrics["macro_f1"]:.4f}')

print('\nClassification Report:')
print(classification_report(
    y_test_cls, pred_test_cls,
    target_names=['Down', 'Stable', 'Up'], zero_division=0
))

df_test_pred['predicted_next_week_direction'] = [
    INVERSE_DIRECTION_LABEL_MAP.get(p, str(p)) for p in pred_test_cls
]
df_test_pred['direction_correct'] = (
    df_test_pred['next_week_direction'] == df_test_pred['predicted_next_week_direction']
).astype(int)

ensure_dir('outputs/model_results/')
safe_to_csv(df_test_pred, 'outputs/model_results/predictions_test.csv')
print(f'\nPrediksi test disimpan: outputs/model_results/predictions_test.csv')
print(f'Shape: {df_test_pred.shape}')


=== Test Classification Metrics ===
  Accuracy       : 0.7692
  Macro Precision: 0.6650
  Macro Recall   : 0.6371
  Macro F1       : 0.6478

Classification Report:
              precision    recall  f1-score   support

        Down       0.43      0.32      0.37      2547
      Stable       0.82      0.87      0.84     13976
          Up       0.74      0.73      0.73      5077

    accuracy                           0.77     21600
   macro avg       0.66      0.64      0.65     21600
weighted avg       0.76      0.77      0.76     21600


Prediksi test disimpan: outputs/model_results/predictions_test.csv
Shape: (21600, 11)


## CELL 09.9 — Feature Importance

In [9]:
# CELL 09.9 — Feature Importance

ensure_dir('outputs/model_results/')
df_fi_reg = None
df_fi_cls = None

def get_importance(model, feature_cols, model_name):
    if hasattr(model, 'feature_importances_'):
        return pd.DataFrame({
            'feature': feature_cols,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False).reset_index(drop=True)
    elif hasattr(model, 'coef_'):
        coef = model.coef_.flatten() if model.coef_.ndim > 1 else model.coef_
        return pd.DataFrame({
            'feature': feature_cols,
            'importance': np.abs(coef)
        }).sort_values('importance', ascending=False).reset_index(drop=True)
    else:
        print(f'  [INFO] {model_name} tidak memiliki feature importance.')
        return None

df_fi_reg = get_importance(best_regressor, FEATURE_COLS, best_reg_name)
if df_fi_reg is not None:
    safe_to_csv(df_fi_reg, 'outputs/model_results/feature_importance_regressor.csv')
    print('Feature importance regressor disimpan.')
    print(df_fi_reg.head(10).to_string(index=False))

df_fi_cls = get_importance(best_classifier, FEATURE_COLS, best_cls_name)
if df_fi_cls is not None:
    safe_to_csv(df_fi_cls, 'outputs/model_results/feature_importance_classifier.csv')
    print('\nFeature importance classifier disimpan.')
    print(df_fi_cls.head(10).to_string(index=False))


Feature importance regressor disimpan.
            feature  importance
       lag_1_income    0.423948
    rolling_mean_4w    0.187612
    rolling_mean_8w    0.092991
    rolling_mean_2w    0.089400
     rolling_max_4w    0.027749
gig_content_creator    0.007809
   income_growth_1w    0.007793
 gig_pekerja_harian    0.006975
  rolling_median_4w    0.006845
     rolling_std_8w    0.006721

Feature importance classifier disimpan.
                   feature  importance
             rolling_cv_4w    0.124641
         income_volatility    0.109071
rolling_last_vs_median_pct    0.076754
       lag_ratio_1_to_mean    0.063356
            trend_slope_4w    0.056036
    last_income_change_pct    0.055513
       income_trend_4w_pct    0.049288
            rolling_min_4w    0.046762
          income_growth_1w    0.046350
   is_previous_week_stable    0.036686


## CELL 09.10 — Visualization

In [10]:
# CELL 09.10 — Visualization

ensure_dir('outputs/charts/')

# 1. Actual vs Predicted
fig, ax = plt.subplots(figsize=(8, 6))
sample_n = min(500, len(y_test_reg))
idx = np.random.choice(len(y_test_reg), sample_n, replace=False)
ax.scatter(y_test_reg[idx]/1e6, pred_test_idr[idx]/1e6,
           alpha=0.4, s=20, color='steelblue', label='Prediksi')
mn = min(y_test_reg[idx].min(), pred_test_idr[idx].min()) / 1e6
mx = max(y_test_reg[idx].max(), pred_test_idr[idx].max()) / 1e6
ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Ideal')
ax.set_xlabel('Actual Income (juta Rp)')
ax.set_ylabel('Predicted Income (juta Rp)')
ax.set_title(
    f'Actual vs Predicted — {best_reg_name}\n'
    f'MAE={fmt_idr(reg_test_metrics["mae"])}  R2={reg_test_metrics["r2"]:.4f}'
)
ax.legend()
safe_savefig('outputs/charts/regression_prediction_vs_actual.png')

# 2. Residual Distribution
residuals = y_test_reg - pred_test_idr
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(residuals / 1e6, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', lw=1.5, label='Zero')
ax.set_xlabel('Residual (juta Rp)')
ax.set_ylabel('Frekuensi')
ax.set_title(f'Residual Distribution — {best_reg_name}')
ax.legend()
safe_savefig('outputs/charts/regression_residual_distribution.png')

# 3. Confusion Matrix
cm = confusion_matrix(y_test_cls, pred_test_cls)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down', 'Stable', 'Up'],
            yticklabels=['Down', 'Stable', 'Up'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(
    f'Confusion Matrix — {best_cls_name}\n'
    f'Acc={cls_test_metrics["accuracy"]:.4f}  Macro F1={cls_test_metrics["macro_f1"]:.4f}'
)
safe_savefig('outputs/charts/classification_confusion_matrix.png')

# 4. Feature Importance Top 20
if df_fi_reg is not None:
    top20 = df_fi_reg.head(20)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top20['feature'][::-1], top20['importance'][::-1],
            color='steelblue', alpha=0.85)
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 20 Feature Importance — {best_reg_name}')
    plt.tight_layout()
    safe_savefig('outputs/charts/feature_importance_best_model.png')

# 5. Error by gig_type
if 'gig_type' in df_test_pred.columns:
    gig_err = df_test_pred.groupby('gig_type')['absolute_error'].mean().sort_values()
    fig, ax = plt.subplots(figsize=(9, 5))
    gig_err.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
    ax.set_xlabel('Mean Absolute Error (Rp)')
    ax.set_title(f'Regression MAE by Gig Type — {best_reg_name}')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp {x:,.0f}'))
    plt.tight_layout()
    safe_savefig('outputs/charts/regression_error_by_gig_type.png')
else:
    print('[INFO] Kolom gig_type tidak tersedia, chart dilewati.')


Chart disimpan: outputs/charts/regression_prediction_vs_actual.png
Chart disimpan: outputs/charts/regression_residual_distribution.png
Chart disimpan: outputs/charts/classification_confusion_matrix.png
Chart disimpan: outputs/charts/feature_importance_best_model.png
Chart disimpan: outputs/charts/regression_error_by_gig_type.png


## CELL 09.11 — Save Models dan Metrics

In [11]:
# CELL 09.11 — Save Models dan Metrics

ensure_dir('outputs/model_results/')

with open('outputs/model_results/best_income_regressor.pkl', 'wb') as f:
    pickle.dump(best_regressor, f)
print('Disimpan: outputs/model_results/best_income_regressor.pkl')

with open('outputs/model_results/best_direction_classifier.pkl', 'wb') as f:
    pickle.dump(best_classifier, f)
print('Disimpan: outputs/model_results/best_direction_classifier.pkl')

# Regression metrics CSV
df_reg_out = df_reg_val[['model_name','mae','rmse','mape','r2']].copy()
df_reg_out.rename(columns={'mae':'val_mae','rmse':'val_rmse','mape':'val_mape','r2':'val_r2'}, inplace=True)
test_reg_row = pd.DataFrame([{
    'model_name': f'{best_reg_name} [TEST]',
    'val_mae': reg_test_metrics['mae'],
    'val_rmse': reg_test_metrics['rmse'],
    'val_mape': reg_test_metrics['mape'],
    'val_r2': reg_test_metrics['r2'],
}])
df_reg_all = pd.concat([df_reg_out, test_reg_row], ignore_index=True)
safe_to_csv(df_reg_all, 'outputs/model_results/regression_metrics.csv')
print('Disimpan: outputs/model_results/regression_metrics.csv')

# Classification metrics CSV
df_cls_out = df_cls_val[['model_name','accuracy','macro_precision','macro_recall','macro_f1']].copy()
df_cls_out.rename(columns={
    'accuracy':'val_accuracy','macro_precision':'val_macro_precision',
    'macro_recall':'val_macro_recall','macro_f1':'val_macro_f1'
}, inplace=True)
test_cls_row = pd.DataFrame([{
    'model_name': f'{best_cls_name} [TEST]',
    'val_accuracy': cls_test_metrics['accuracy'],
    'val_macro_precision': cls_test_metrics['macro_precision'],
    'val_macro_recall': cls_test_metrics['macro_recall'],
    'val_macro_f1': cls_test_metrics['macro_f1'],
}])
df_cls_all = pd.concat([df_cls_out, test_cls_row], ignore_index=True)
safe_to_csv(df_cls_all, 'outputs/model_results/classification_metrics.csv')
print('Disimpan: outputs/model_results/classification_metrics.csv')

# Training metadata
training_metadata = {
    'best_regressor_name': best_reg_name,
    'best_classifier_name': best_cls_name,
    'feature_count': len(FEATURE_COLS),
    'train_rows': int(len(df_train)),
    'val_rows': int(len(df_val)),
    'test_rows': int(len(df_test)),
    'regression_test_metrics': {
        'mae': float(reg_test_metrics['mae']),
        'rmse': float(reg_test_metrics['rmse']),
        'mape': float(reg_test_metrics['mape']),
        'r2': float(reg_test_metrics['r2']),
    },
    'classification_test_metrics': {
        'accuracy': float(cls_test_metrics['accuracy']),
        'macro_precision': float(cls_test_metrics['macro_precision']),
        'macro_recall': float(cls_test_metrics['macro_recall']),
        'macro_f1': float(cls_test_metrics['macro_f1']),
    },
    'target_regression': 'next_week_income',
    'target_classification': 'next_week_direction',
    'model_type_note': 'Baseline ML model, not final LSTM. Digunakan untuk benchmark AI Engineer.',
    'direction_label_map': DIRECTION_LABEL_MAP,
    'feature_columns': FEATURE_COLS,
    'log1p_transform': True,
    'split_strategy': 'by synthetic_user_id, bukan random row',
    'anti_leakage_check': 'PASSED',
    'random_state': 42,
}

with open('outputs/model_results/training_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(training_metadata, f, indent=2, ensure_ascii=False)
print('Disimpan: outputs/model_results/training_metadata.json')


Disimpan: outputs/model_results/best_income_regressor.pkl
Disimpan: outputs/model_results/best_direction_classifier.pkl
Disimpan: outputs/model_results/regression_metrics.csv
Disimpan: outputs/model_results/classification_metrics.csv
Disimpan: outputs/model_results/training_metadata.json


## CELL 09.12 — Generate Model Evaluation Report

In [12]:
# CELL 09.12 — Generate Model Evaluation Report

top_fi_lines = ''
lag_important = '_Tidak dapat dievaluasi._'

if df_fi_reg is not None:
    top_fi_lines = '\n'.join(
        f'| {row["feature"]} | {row["importance"]:.6f} |'
        for _, row in df_fi_reg.head(10).iterrows()
    )
    lag_feats = df_fi_reg[
        df_fi_reg['feature'].str.contains('lag|rolling|shift', case=False)
    ]
    lag_important = (', '.join(lag_feats.head(5)['feature'].tolist())
                     if len(lag_feats) > 0
                     else '_Tidak ditemukan fitur lag/rolling dalam top importance._')
else:
    top_fi_lines = '_Feature importance tidak tersedia untuk model ini._'

reg_table = '| Model | Val MAE | Val RMSE | Val MAPE | Val R2 |\n|---|---|---|---|---|\n'
for _, row in df_reg_val.iterrows():
    reg_table += (f'| {row["model_name"]} | {fmt_idr(row["mae"])} | '
                  f'{fmt_idr(row["rmse"])} | {row["mape"]:.2f}% | {row["r2"]:.4f} |\n')

cls_table = '| Model | Val Accuracy | Val Prec | Val Recall | Val F1 |\n|---|---|---|---|---|\n'
for _, row in df_cls_val.iterrows():
    cls_table += (f'| {row["model_name"]} | {row["accuracy"]:.4f} | '
                  f'{row["macro_precision"]:.4f} | {row["macro_recall"]:.4f} | '
                  f'{row["macro_f1"]:.4f} |\n')

r2_note = ('sudah cukup baik sebagai acuan awal'
           if reg_test_metrics['r2'] > 0.5
           else 'masih perlu peningkatan signifikan')

report_text = (
    '# Model Evaluation Report\n'
    '**Fingo Income Predictor** | Tim CC26-PSU217 | DS2 Clarisya Adeline\n\n'
    '---\n\n'
    '## 1. Overview\n'
    'Notebook ini melatih dan mengevaluasi model baseline ML untuk prediksi pendapatan mingguan '
    'pekerja gig economy. Baseline digunakan sebagai benchmark sebelum AI Engineer '
    'mengembangkan model sequence/LSTM.\n\n'
    '---\n\n'
    '## 2. Dataset Split Summary\n\n'
    '| Split | Baris |\n|---|---|\n'
    f'| Train | {len(df_train):,} |\n'
    f'| Validation | {len(df_val):,} |\n'
    f'| Test | {len(df_test):,} |\n\n'
    'Split dilakukan **by `synthetic_user_id`**, bukan random row — tidak ada overlap user antar subset.\n\n'
    '---\n\n'
    '## 3. Feature Count\n\n'
    f'Jumlah fitur: **{len(FEATURE_COLS)} fitur**  \n'
    'Anti-leakage check: **PASSED**\n\n'
    '---\n\n'
    '## 4. Regression Model Comparison (Validation)\n\n'
    f'{reg_table}\n'
    '---\n\n'
    '## 5. Best Regression Model\n\n'
    f'**{best_reg_name}** — dipilih berdasarkan MAE terkecil di validation set.\n\n'
    '---\n\n'
    '## 6. Regression Test Metrics\n\n'
    '| Metric | Value |\n|---|---|\n'
    f'| MAE | {fmt_idr(reg_test_metrics["mae"])} |\n'
    f'| RMSE | {fmt_idr(reg_test_metrics["rmse"])} |\n'
    f'| MAPE | {reg_test_metrics["mape"]:.2f}% |\n'
    f'| R2 | {reg_test_metrics["r2"]:.4f} |\n\n'
    '---\n\n'
    '## 7. Classification Model Comparison (Validation)\n\n'
    f'{cls_table}\n'
    '---\n\n'
    '## 8. Best Classification Model\n\n'
    f'**{best_cls_name}** — dipilih berdasarkan macro F1 tertinggi di validation set.\n\n'
    '---\n\n'
    '## 9. Classification Test Metrics\n\n'
    '| Metric | Value |\n|---|---|\n'
    f'| Accuracy | {cls_test_metrics["accuracy"]:.4f} |\n'
    f'| Macro Precision | {cls_test_metrics["macro_precision"]:.4f} |\n'
    f'| Macro Recall | {cls_test_metrics["macro_recall"]:.4f} |\n'
    f'| Macro F1 | {cls_test_metrics["macro_f1"]:.4f} |\n\n'
    '---\n\n'
    '## 10. Top Feature Importance (Best Regressor)\n\n'
    '| Feature | Importance |\n|---|---|\n'
    f'{top_fi_lines}\n\n'
    '---\n\n'
    '## 11. Interpretasi\n\n'
    f'- **MAE** model terbaik di test set adalah **{fmt_idr(reg_test_metrics["mae"])}**.\n'
    f'- **MAPE** sebesar **{reg_test_metrics["mape"]:.2f}%** menunjukkan rata-rata error relatif terhadap income aktual.\n'
    f'- Model baseline {r2_note} (R2 = {reg_test_metrics["r2"]:.4f}).\n'
    '- Model sequence seperti **LSTM sangat layak dicoba** karena data income bersifat temporal dan berurutan per user.\n'
    f'- Fitur lag/rolling yang penting: {lag_important}\n'
    '- Split sudah dilakukan by user — tidak ada data leakage antar split.\n'
    '- Anti-leakage check sudah diverifikasi sebelum training.\n\n'
    '---\n\n'
    '## 12. Rekomendasi untuk AI Engineer\n\n'
    '1. Gunakan baseline ini (MAE, RMSE, MAPE, R2) sebagai benchmark minimum yang harus dilewati LSTM.\n'
    '2. Pertahankan split by `synthetic_user_id` agar evaluasi tetap fair.\n'
    '3. Load `outputs/model_contract/income_scalers.pkl` untuk scaler yang konsisten.\n'
    '4. Load `outputs/model_contract/feature_columns.json` untuk daftar fitur.\n'
    '5. Target regression: `next_week_income` (latih log1p, evaluasi rupiah asli).\n'
    '6. Target classification: `next_week_direction` (Down=0, Stable=1, Up=2).\n'
    '7. Perhatikan distribusi kelas yang tidak seimbang pada classification.\n'
    '8. Model terbaik tersimpan di `outputs/model_results/` siap untuk perbandingan.\n'
)

ensure_dir('outputs/model_results/')
with open('outputs/model_results/model_evaluation_report.md', 'w', encoding='utf-8') as f:
    f.write(report_text)
print('Disimpan: outputs/model_results/model_evaluation_report.md')


Disimpan: outputs/model_results/model_evaluation_report.md


## CELL 09.13 — Final Validation

In [13]:
# CELL 09.13 — Final Validation

expected_outputs = [
    'outputs/model_results/best_income_regressor.pkl',
    'outputs/model_results/best_direction_classifier.pkl',
    'outputs/model_results/regression_metrics.csv',
    'outputs/model_results/classification_metrics.csv',
    'outputs/model_results/predictions_test.csv',
    'outputs/model_results/model_evaluation_report.md',
    'outputs/model_results/training_metadata.json',
    'outputs/charts/regression_prediction_vs_actual.png',
    'outputs/charts/regression_residual_distribution.png',
    'outputs/charts/classification_confusion_matrix.png',
    'outputs/charts/feature_importance_best_model.png',
]

print('=== Final Output Validation ===')
all_ok = True
for path in expected_outputs:
    exists = os.path.isfile(path)
    size   = os.path.getsize(path) if exists else 0
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {path}  ({size:,} bytes)')
    if not exists:
        all_ok = False

for opt_path in ['outputs/charts/regression_error_by_gig_type.png',
                 'outputs/model_results/feature_importance_regressor.csv',
                 'outputs/model_results/feature_importance_classifier.csv']:
    if os.path.isfile(opt_path):
        print(f'  [OK]  {opt_path}  ({os.path.getsize(opt_path):,} bytes) [opsional]')

if all_ok:
    print('\nSemua output berhasil dibuat. Notebook 09 selesai.')
else:
    print('\n[WARNING] Beberapa output tidak ditemukan. Periksa cell sebelumnya.')


=== Final Output Validation ===
  [OK] outputs/model_results/best_income_regressor.pkl  (808,533 bytes)
  [OK] outputs/model_results/best_direction_classifier.pkl  (49,262,188 bytes)
  [OK] outputs/model_results/regression_metrics.csv  (599 bytes)
  [OK] outputs/model_results/classification_metrics.csv  (567 bytes)
  [OK] outputs/model_results/predictions_test.csv  (2,149,733 bytes)
  [OK] outputs/model_results/model_evaluation_report.md  (3,694 bytes)
  [OK] outputs/model_results/training_metadata.json  (2,421 bytes)
  [OK] outputs/charts/regression_prediction_vs_actual.png  (97,644 bytes)
  [OK] outputs/charts/regression_residual_distribution.png  (36,717 bytes)
  [OK] outputs/charts/classification_confusion_matrix.png  (52,589 bytes)
  [OK] outputs/charts/feature_importance_best_model.png  (91,267 bytes)
  [OK]  outputs/charts/regression_error_by_gig_type.png  (50,129 bytes) [opsional]
  [OK]  outputs/model_results/feature_importance_regressor.csv  (1,725 bytes) [opsional]
  [OK]  o

## CELL 09.14 — Git Push Footer

In [14]:
# GIT PUSH — Commit dan push output notebook ini ke GitHub
import os, subprocess

LOCAL_DIR     = '/content/fingo-income-analysis'
BRANCH_NAME   = 'feat/income-predictor-final'
NOTEBOOK_NAME = '09_Model_Training_Evaluation.ipynb'

os.chdir(LOCAL_DIR)

def run_cmd(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f'$ {cmd}')
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f'Command gagal: {cmd}')
    return r

run_cmd('git config user.email "adelineclarisya@gmail.com"')
run_cmd('git config user.name "ClarisyaA"')

print('\n[1] Cek status')
run_cmd('git status --short', check=False)

print('\n[2] Add semua perubahan output')
run_cmd('git add data/ outputs/ notebooks/ *.ipynb', check=False)

print('\n[3] Commit')
commit_result = run_cmd(
    'git commit -m "feat(DS2): train and evaluate baseline income models"',
    check=False
)
if commit_result.returncode != 0:
    print('[INFO] Tidak ada perubahan baru, skip commit.')

print('\n[4] Fetch remote terbaru')
run_cmd('git fetch origin')

print('\n[5] Rebase lalu push')
run_cmd(f'git pull --rebase origin {BRANCH_NAME}')
run_cmd(f'git push -u origin {BRANCH_NAME}')

print('\nPush berhasil!')


$ git config user.email "adelineclarisya@gmail.com"
$ git config user.name "ClarisyaA"

[1] Cek status
$ git status --short
?? outputs/charts/classification_confusion_matrix.png
?? outputs/charts/feature_importance_best_model.png
?? outputs/charts/regression_error_by_gig_type.png
?? outputs/charts/regression_prediction_vs_actual.png
?? outputs/charts/regression_residual_distribution.png
?? outputs/model_results/

[2] Add semua perubahan output
$ git add data/ outputs/ notebooks/ *.ipynb

[3] Commit
$ git commit -m "feat(DS2): train and evaluate baseline income models"
[feat/income-predictor-final 13cef82] feat(DS2): train and evaluate baseline income models
 14 files changed, 21949 insertions(+)
 create mode 100644 outputs/charts/classification_confusion_matrix.png
 create mode 100644 outputs/charts/feature_importance_best_model.png
 create mode 100644 outputs/charts/regression_error_by_gig_type.png
 create mode 100644 outputs/charts/regression_prediction_vs_actual.png
 create mode 100